In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [11]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F

class ScratchMCQSolver(nn.Module): 
    def __init__(self, input_dim, hidden_dim=128): 
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 5)

    def forward(self, x): 
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits 
vocab_size = 5000 
model_scratch = ScratchMCQSolver(input_dim=vocab_size)
print(model_scratch)

ScratchMCQSolver(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
)


In [12]:
import pandas as pd 
import numpy as np 
import re 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [13]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [15]:
train_df.fillna("None", inplace=True)

In [16]:
def clean_text(text): 
    text = str(text).lower()
    text = re.sub(r'\s{2,}', '', text)
    return text.strip()

columns_to_clean = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in columns_to_clean: 
    if col in train_df.columns: 
        train_df[col] = train_df[col].apply(clean_text)

print("Data Cleaned")

Data Cleaned


In [17]:
vectorizer = TfidfVectorizer(stop_words='english')
predictions = []
actuals = []

for idx, row in train_df.iterrows():
    corpus = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]

    tfidf_matrix = vectorizer.fit_transform(corpus)
    prompt_vector = tfidf_matrix[0:1]
    options_matrix = tfidf_matrix[1:]

    similarities = cosine_similarity(prompt_vector, options_matrix).flatten()

    labels = ['A', 'B', 'C', 'D', 'E']
    top_3_idx = similarities.argsort()[-3:][::-1]
    top_3_preds = [labels[i] for i in top_3_idx]

    predictions.append(top_3_preds)
    if 'answer' in train_df.columns: 
        actuals.append(row['answer'])


In [18]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual)+1)

    return 0

if actuals: 
    map_3_score = np.mean([apk(a, p) for a, p in zip(actuals, predictions)])
    print(f'Baseline TF-IDF MAP@3: {map_3_score: .4f}')
else: 
    print('No answer column found')

Baseline TF-IDF MAP@3:  0.3260


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [4]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True